In [40]:
import os
import pandas as pd
import tomllib
from pathlib import Path
import ast

In [14]:
# Open config file 
with open(Path.cwd().parent / "config.toml", "rb") as f:
    cfg = tomllib.load(f)

grid_intensity = cfg["grid_intensity"]
carbon_price = cfg["carbon_price"]
energy = cfg["energy_wh_per_1k_queries"]


In [26]:
routerbench = pd.read_pickle("data/raw/routerbench_0shot.pkl")
print(routerbench.shape)
print(routerbench.columns.tolist())

# columns to filter on 
keep_cols =[
    "sample_id", "prompt", "eval_name",
    "mistralai/mistral-7b-chat",  # Quality score 
    "mistralai/mistral-7b-chat|model_response",
    "mistralai/mistral-7b-chat|total_cost",
    "mistralai/mixtral-8x7b-chat",
    "mistralai/mixtral-8x7b-chat|model_response",
    "mistralai/mixtral-8x7b-chat|total_cost",
    "zero-one-ai/Yi-34B-Chat",
    "zero-one-ai/Yi-34B-Chat|model_response",
    "zero-one-ai/Yi-34B-Chat|total_cost",
    "meta/llama-2-70b-chat",
    "meta/llama-2-70b-chat|model_response",
    "meta/llama-2-70b-chat|total_cost",
]
routerbench = routerbench[keep_cols]

# Create a task column that only specifies main parent group of eval_name 
# to help later with stratified sampling
routerbench["task"] = routerbench["eval_name"].str.split("-").str[0]
# Rename grade to gsm8k to be consistent with the paper
routerbench["task"] = routerbench["task"].replace({"grade": "gsm8k"})
# Keep tasks that are central to the Routerbench paper and cover 
# knowledge, commonsense, math, and code 
TASKS = ["mmlu", "hellaswag", "arc", "gsm8k", "mbpp"]
# Randomly select n_prompts 
n_prompts = cfg["n_prompts"]
seed = cfg["seed"]
# Select all mbpp tasks since there's the fewest (427)(and less than n_prompts/5)
mbpp = routerbench[routerbench["task"] == "mbpp"]                      
n_per_task = (n_prompts - len(mbpp)) // (len(TASKS) - 1)  # (2500 - 427) // 4 = 518

rb_sampled = pd.concat([
    routerbench[routerbench["task"].isin(TASKS[:-1])].groupby("task").sample(n=n_per_task, random_state=seed),
    mbpp,
])


(36497, 37)
['sample_id', 'prompt', 'eval_name', 'WizardLM/WizardLM-13B-V1.2', 'claude-instant-v1', 'claude-v1', 'claude-v2', 'gpt-3.5-turbo-1106', 'gpt-4-1106-preview', 'meta/code-llama-instruct-34b-chat', 'meta/llama-2-70b-chat', 'mistralai/mistral-7b-chat', 'mistralai/mixtral-8x7b-chat', 'zero-one-ai/Yi-34B-Chat', 'gpt-3.5-turbo-1106|model_response', 'claude-instant-v1|model_response', 'claude-v1|model_response', 'claude-v2|model_response', 'gpt-4-1106-preview|model_response', 'meta/llama-2-70b-chat|model_response', 'mistralai/mixtral-8x7b-chat|model_response', 'zero-one-ai/Yi-34B-Chat|model_response', 'WizardLM/WizardLM-13B-V1.2|model_response', 'meta/code-llama-instruct-34b-chat|model_response', 'mistralai/mistral-7b-chat|model_response', 'gpt-3.5-turbo-1106|total_cost', 'claude-instant-v1|total_cost', 'claude-v1|total_cost', 'claude-v2|total_cost', 'gpt-4-1106-preview|total_cost', 'meta/llama-2-70b-chat|total_cost', 'mistralai/mixtral-8x7b-chat|total_cost', 'zero-one-ai/Yi-34B-Ch

In [28]:
MODELS = [
    "mistralai/mistral-7b-chat",
    "mistralai/mixtral-8x7b-chat",
    "zero-one-ai/Yi-34B-Chat",
    "meta/llama-2-70b-chat",
]

# Reshape so that each row is a prompt, model pair which simplifies calculations
prompt_model_pair = pd.concat([
    rb_sampled[["sample_id", "prompt", "eval_name", "task"]].assign(
        model=m,
        quality=rb_sampled[m],
        response=rb_sampled[f"{m}|model_response"],
        cost=rb_sampled[f"{m}|total_cost"],
    )
    for m in MODELS
], ignore_index=True)

prompt_model_pair 

,sample_id,prompt,eval_name,task,model,quality,response,cost
0,arc-challenge.test.881,['Which statement about the molecules in ice a...,arc-challenge,arc,mistralai/mistral-7b-chat,1.0,['C)'],0.000022
1,arc-challenge.test.1163,['A dialysis machine can be used to remove was...,arc-challenge,arc,mistralai/mistral-7b-chat,1.0,['A)'],0.000015
2,arc-challenge.val.143,['Which action creates competition between env...,arc-challenge,arc,mistralai/mistral-7b-chat,0.0,['A)'],0.000015
3,arc-challenge.test.1058,['A population of deer recently increased in s...,arc-challenge,arc,mistralai/mistral-7b-chat,0.0,['A)'],0.000016
4,arc-challenge.test.141,['Which of the following manufactured products...,arc-challenge,arc,mistralai/mistral-7b-chat,0.0,['A)'],0.000013
...,...,...,...,...,...,...,...,...
9991,mbpp.dev.95,['Write a function to determine if the sum of ...,mbpp,mbpp,meta/llama-2-70b-chat,0.0,"["" Here's a possible implementation of a funct...",0.000246
9992,mbpp.dev.96,['Write a function to count the number of char...,mbpp,mbpp,meta/llama-2-70b-chat,0.0,[' Here\'s a possible implementation of such a...,0.000397
9993,mbpp.dev.97,['Write a function that counts the number of p...,mbpp,mbpp,meta/llama-2-70b-chat,0.0,"["" Sure! Here's a possible implementation of s...",0.000371
9994,mbpp.dev.98,['Write a python function to find the smallest...,mbpp,mbpp,meta/llama-2-70b-chat,0.0,"["" Sure! Here's a Python function that finds t...",0.000315


In [41]:
# Extract response text and calculate token length
text = prompt_model_pair["response"].apply(ast.literal_eval).str[0]
prompt_model_pair["response_length"] = text.str.split().str.len()